# MobileNetV3 + Optical Flow (CASME II)

Notebook para entrenamiento de microexpresiones usando secuencias `.npy` (optical flow) y MobileNetV3-Small.

Ventajas sobre ResNet50:
- Solo 1.5M parámetros (vs 25.5M en ResNet50)
- Ratio datos:parámetros = 1:3,600 (vs 1:59,000)
- Diseñado para datasets pequeños

In [ ]:
import os
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from PIL import Image
import csv
from pathlib import Path

In [ ]:
from pathlib import Path
import torch

# CONFIG
BASE_DIR = Path("Microexpresiones")

DATA_DIRS = [
    "../outputs_dataset_index",
    "../output_extraction_meme"
]

CSV_INDEXES = [
    "../outputs_dataset_index/extraction_index.csv",
    "../output_extraction_meme/extraction_index.csv"
]

NUM_FRAMES = 16
BATCH_SIZE = 8  # MobileNetV3 puede usar batch más grande
EPOCHS = 30
LR = 1e-4
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print("Using device:", DEVICE)

In [ ]:
# DATA AUGMENTATION - AGRESIVA (igual que ResNet50 optimizado)
transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.RandomHorizontalFlip(p=0.7),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2),
    transforms.RandomRotation(15),
    transforms.RandomAffine(degrees=0, translate=(0.1, 0.1)),
    transforms.GaussianBlur(kernel_size=3, sigma=(0.1, 2.0)),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
])

print("✅ Data augmentation agresiva activada")

In [ ]:
class FlowDataset(Dataset):
    def __init__(self, data_dirs, csv_files, transform, num_frames=16):
        self.transform = transform
        self.num_frames = num_frames
        self.samples = []

        # 🔹 Leer múltiples datasets
        for data_dir, csv_index in zip(data_dirs, csv_files):
            data_dir = Path(data_dir)

            with open(csv_index, 'r', encoding='utf-8') as f:
                reader = csv.DictReader(f)
                for row in reader:
                    rel = row['archivo'].replace("\\", "/")  # FIX rutas
                    path = data_dir / rel
                    label = row['emocion'].strip().lower()

                    # normalización de nombres
                    label_alias = {
                        "felicidad": "felicidad",
                        "feliz": "felicidad",
                    
                        "enojo": "enojo",
                        "ira": "enojo",
                    
                        "miedo": "miedo",
                    
                        "tristeza": "tristeza",
                    
                        "sorpresa": "sorpresa",
                    
                        "asco": "asco",
                    
                        "represion": "otros",
                        "otros": "otros"
                    }
                    
                    label = label_alias.get(label, label)
                    subject = row.get('sujeto', 'unknown')

                    if path.exists():
                        self.samples.append((path, label, subject))

        # 🔹 Mapear etiquetas globales (IMPORTANTE al mezclar datasets)
        self.labels = sorted(list(set([s[1] for s in self.samples])))
        self.label_map = {l:i for i,l in enumerate(self.labels)}

    def __len__(self):
        return len(self.samples)

    def sample_frames(self, seq):
        n = seq.shape[0]
        idx = np.linspace(0, n-1, self.num_frames).astype(int)
        return seq[idx]

    def __getitem__(self, idx):
        path, label, _ = self.samples[idx]
        seq = np.load(path)

        frames = self.sample_frames(seq)

        imgs = []
        for f in frames:
            # 🔹 Normalización específica de tu flow
            f0 = np.clip((f[...,0]+1)/2,0,1)
            f1 = np.clip((f[...,1]+1)/2,0,1)
            f2 = np.clip(f[...,2],0,1)

            rgb = np.stack([f0,f1,f2],axis=-1)
            rgb = (rgb*255).astype(np.uint8)

            imgs.append(self.transform(Image.fromarray(rgb)))

        return torch.stack(imgs), self.label_map[label]

In [ ]:
import torch
import torch.nn as nn
from torchvision import models

class MobileNetV3Flow(nn.Module):
    def __init__(self, num_classes):
        super().__init__()

        # MobileNetV3-Small preentrenado
        mobilenet = models.mobilenet_v3_small(weights=models.MobileNet_V3_Small_Weights.DEFAULT)

        # Backbone (features extractoras sin cabeza de clasificación)
        self.backbone = mobilenet.features

        # 🔹 FINE-TUNING: Backbone congelado, solo clasificador entrenable
        # MobileNetV3 ya está optimizado, no necesitamos descongelar capas
        for param in self.backbone.parameters():
            param.requires_grad = False

        # Pool espacial
        self.pool = nn.AdaptiveAvgPool2d(1)

        # MobileNetV3-Small tiene 576 features finales
        # Clasificador simple (2 capas)
        self.classifier = nn.Sequential(
            nn.Linear(576, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        B, T, C, H, W = x.shape

        # (B,T,C,H,W) → (B*T,C,H,W)
        x = x.view(B*T, C, H, W)

        # Backbone CNN (extrae features)
        feat = self.backbone(x)

        # Pool espacial → (B*T,576)
        feat = self.pool(feat).flatten(1)

        # Volver a secuencia → (B,T,576)
        feat = feat.view(B, T, -1)

        # 🔹 Pool temporal (CLAVE)
        feat = feat.max(dim=1).values   # (B,576)

        # Clasificación final
        out = self.classifier(feat)

        return out

In [ ]:
import csv
from pathlib import Path

DATA_DIRS = [
    Path("../outputs_dataset_index"),
    Path("../output_extraction_meme")
]

CSV_INDEXES = [
    "../outputs_dataset_index/extraction_index.csv",
    "../output_extraction_meme/extraction_index.csv"
]

for data_dir, csv_file in zip(DATA_DIRS, CSV_INDEXES):
    print(f"\n=== Revisando dataset: {data_dir} ===\n")

    with open(csv_file, 'r', encoding='utf-8') as f:
        reader = csv.DictReader(f)
        for i, row in enumerate(reader):
            rel = row['archivo'].replace("\\", "/")  # 🔥 mismo fix
            full = data_dir / rel

            print("CSV:", rel)
            print("FULL:", full)
            print("EXISTS:", full.exists())
            print("-"*50)

            if i == 5:
                break

In [ ]:
from torch.utils.data import DataLoader, WeightedRandomSampler
import numpy as np

# =========================
# LOAD DATA (CORRECTO)
# =========================
dataset = FlowDataset(
    data_dirs=DATA_DIRS,
    csv_files=CSV_INDEXES,
    transform=transform,
    num_frames=NUM_FRAMES
)

# eliminar "otros"
dataset.samples = [s for s in dataset.samples if s[1] != "otros"]

labels_unique = sorted(list(set([s[1] for s in dataset.samples])))
dataset.label_map = {l:i for i,l in enumerate(labels_unique)}

num_classes = len(dataset.label_map)
print("Classes:", dataset.label_map)

# =========================
# OVERSAMPLING: Duplicar clases minoritarias
# =========================
labels_all = np.array([dataset.label_map[s[1]] for s in dataset.samples])
counts = np.bincount(labels_all)
max_count = counts.max()

oversampled_idx = []
for i in range(len(dataset.samples)):
    label = labels_all[i]
    # Duplicar según la brecha con la clase mayoritaria
    times = int(max_count / counts[label])
    oversampled_idx.extend([i] * times)

print(f"\n📊 Oversampling aplicado:")
print(f"  Antes: {len(dataset.samples)} muestras")
print(f"  Después: {len(oversampled_idx)} muestras")
print(f"  Ratio de aumento: {len(oversampled_idx)/len(dataset.samples):.2f}x")

# =========================
# SAMPLER CON OVERSAMPLING
# =========================
labels = [labels_all[i] for i in oversampled_idx]
counts = np.bincount(labels)

weights = 1.0 / (counts + 1e-6)
sample_weights = [weights[label] for label in labels]

sampler = WeightedRandomSampler(sample_weights, len(sample_weights))

print(f"\n✅ Dataset balanceado:")
print(f"  Distribución: {np.bincount(labels)}")

print("Probando dataset directo...")
x, y = dataset[0]
print("OK:", x.shape, y)

# =========================
# DATALOADER CON OVERSAMPLING
# =========================
train_loader = DataLoader(
    dataset,
    batch_size=BATCH_SIZE,
    sampler=sampler,
    num_workers=0,      # en notebook
    pin_memory=True     # mejora en GPU
)

In [ ]:
# =========================
# INIT MODEL
# =========================
model = MobileNetV3Flow(num_classes).to(DEVICE)

# Contar parámetros entrenables
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model.parameters())
print(f"\n🔧 Modelo MobileNetV3-Small:")
print(f"  Parámetros entrenables: {trainable_params/1e6:.2f}M / {total_params/1e6:.2f}M")
print(f"  Ratio datos:parámetros = {len(dataset.samples) / total_params:.1f}:1")
print(f"  (vs 0.016:1 en ResNet50, vs 0.038:1 en ResNet18)")

# optimizer (más estable que Adam)
optimizer = torch.optim.AdamW(
    [p for p in model.parameters() if p.requires_grad],
    lr=LR, 
    weight_decay=1e-5
)

# =========================
# CLASS WEIGHTS MÁS ALTOS (DESBALANCE SEVERO)
# =========================
labels = np.array([dataset.label_map[s[1]] for s in dataset.samples])
counts = np.bincount(labels)

# Fórmula mejorada: inverso cuadrado para penalizar más las minoritarias
class_weights = (len(labels) / counts) ** 1.5
class_weights = class_weights / class_weights.mean()

print(f"\n⚖️  Pesos de clases:")
for i, w in enumerate(class_weights):
    print(f"  Clase {i}: {w:.3f}x (n={counts[i]})")

class_weights = torch.tensor(class_weights, dtype=torch.float32).to(DEVICE)

# =========================
# LOSS FINAL CON PESOS AUMENTADOS
# =========================
criterion = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=0.1)

In [ ]:
import numpy as np
import torch
from torch.utils.data import DataLoader, Subset, WeightedRandomSampler

# =========================
# CONFUSION MATRIX
# =========================
def confusion_matrix_np(y_true, y_pred, num_classes):
    cm = np.zeros((num_classes, num_classes), dtype=int)
    for t, p in zip(y_true, y_pred):
        cm[t, p] += 1
    return cm

# =========================
# MACRO F1
# =========================
def macro_f1(cm):
    f1_scores = []

    for c in range(len(cm)):
        tp = cm[c, c]
        fp = cm[:, c].sum() - tp
        fn = cm[c, :].sum() - tp

        precision = tp / (tp + fp) if (tp + fp) > 0 else 0
        recall = tp / (tp + fn) if (tp + fn) > 0 else 0

        if precision + recall == 0:
            f1 = 0
        else:
            f1 = 2 * precision * recall / (precision + recall)

        f1_scores.append(f1)

    return sum(f1_scores) / len(f1_scores)

# =========================
# EVALUATE
# =========================
@torch.no_grad()
def evaluate(model, loader, criterion, num_classes):
    model.eval()

    total_loss = 0
    correct = 0
    total = 0

    y_true = []
    y_pred = []

    for x, y in loader:
        x, y = x.to(DEVICE), y.to(DEVICE)

        out = model(x)
        loss = criterion(out, y)

        total_loss += loss.item() * y.size(0)

        preds = out.argmax(1)

        correct += (preds == y).sum().item()
        total += y.size(0)

        y_true.extend(y.cpu().numpy())
        y_pred.extend(preds.cpu().numpy())

    total_loss /= total
    acc = correct / total

    cm = confusion_matrix_np(y_true, y_pred, num_classes)
    f1 = macro_f1(cm)

    return total_loss, acc, f1, cm


# =========================
# SPLIT ESTRATIFICADO
# =========================
labels = np.array([dataset.label_map[s[1]] for s in dataset.samples])

num_classes = len(np.unique(labels))
train_idx = []
val_idx = []

np.random.seed(42)

for c in range(num_classes):
    idx_c = np.where(labels == c)[0]
    np.random.shuffle(idx_c)

    split = int(0.8 * len(idx_c))

    train_idx.extend(idx_c[:split])
    val_idx.extend(idx_c[split:])

np.random.shuffle(train_idx)
np.random.shuffle(val_idx)

# =========================
# SUBSETS
# =========================
train_dataset = Subset(dataset, train_idx)
val_dataset   = Subset(dataset, val_idx)

# =========================
# SAMPLER BALANCEADO
# =========================
train_labels = [labels[i] for i in train_idx]
counts = np.bincount(train_labels)

weights = 1.0 / (counts + 1e-6)
sample_weights = [weights[l] for l in train_labels]

sampler = WeightedRandomSampler(sample_weights, len(sample_weights))

# =========================
# DATALOADERS
# =========================
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    sampler=sampler,
    num_workers=0  # importante en notebook
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0
)

print(f"Train samples: {len(train_dataset)}")
print(f"Val samples: {len(val_dataset)}")
print("Distribución TRAIN:", np.bincount(train_labels))
print("Batches train:", len(train_loader))


# =========================
# TRAIN LOOP
# =========================
patience = 5
counter = 0
best_f1 = 0

for epoch in range(EPOCHS):
    model.train()

    train_loss = 0
    correct = 0
    total = 0

    print(f"\n🔵 Epoch {epoch+1}/{EPOCHS}")

    for i, (x, y) in enumerate(train_loader):
        x, y = x.to(DEVICE), y.to(DEVICE)

        optimizer.zero_grad()
        out = model(x)
        loss = criterion(out, y)
        loss.backward()
        optimizer.step()

        train_loss += loss.item() * y.size(0)

        preds = out.argmax(1)
        correct += (preds == y).sum().item()
        total += y.size(0)

        # DEBUG opcional
        if i == 0:
            print("Primer batch OK:", x.shape, y.shape)

    train_loss /= total
    train_acc = correct / total

    # =========================
    # VALIDATION
    # =========================
    val_loss, val_acc, val_f1, cm = evaluate(
        model, val_loader, criterion, num_classes
    )

    # =========================
    # GUARDAR MEJOR MODELO
    # =========================
    if val_f1 > best_f1:
        best_f1 = val_f1
        counter = 0

        torch.save({
            "model_state_dict": model.state_dict(),
            "label_map": dataset.label_map
        }, "best_mobilenet_v3.pth")

        print("✅ Modelo mejorado guardado")

    else:
        counter += 1

    # =========================
    # EARLY STOPPING
    # =========================
    '''if counter >= patience:
        print("⛔ Early stopping activado")
        break'''

    # =========================
    # PRINT METRICS
    # =========================
    print(f"""
Train Loss: {train_loss:.4f}
Train Acc : {train_acc:.3f}

Val Loss  : {val_loss:.4f}
Val Acc   : {val_acc:.3f}
Val F1    : {val_f1:.3f}
Best F1   : {best_f1:.3f}
""")

    print("Confusion Matrix:")
    print(cm)